# 3DSSG: 384-D Spatial-Gated Graph Experiment

A separate 100-epoch experiment. It retains the completed baseline object pathway and adds only spatial KNN-gated graph context, cosine learning-rate decay, and edge-loss warm-up.

In [1]:
from pathlib import Path
import argparse

from mtp_pipeline.config import ProjectPaths

SEED = 42
paths = ProjectPaths()
project_root = Path(r"D:\MTP_Project\MTP_Pipeline_3RScan")

# Reuse the completed baseline database: one valid RGB view, pretrained PointNet LiDAR, and global CLIP text.
database_output = paths.output_root / '3rscan_official_static_database_v2'
run_output_root = paths.output_root / 'official_static_spatial_gated_384_v4'
train_scans_file = project_root / 'official_splits' / 'train_scans.txt'
val_scans_file = project_root / 'official_splits' / 'validation_scans.txt'

assert database_output.exists(), f'Missing baseline database: {database_output}'
assert train_scans_file.exists() and val_scans_file.exists()
run_output_root.mkdir(parents=True, exist_ok=True)

def make_train_args(resume_checkpoint=None):
    return argparse.Namespace(
        reference_root=paths.reference_root,
        output_root=run_output_root,
        database=database_output,
        mode='static',
        train_scans=train_scans_file,
        val_scans=val_scans_file,
        split_seed=SEED,
        seed=SEED,
        epochs=100,
        max_scenes=None,
        max_frames=None,
        learning_rate=1e-4,
        embedding_dim=384,
        use_rgb_token_mask=False,
        extended_geometry=False,
        # New graph-only change: geometry-aware messages from the 8 nearest source objects.
        graph_context='spatial_gated',
        graph_knn_neighbors=8,
        # Match the base paper's cosine-decay training style.
        lr_schedule='cosine',
        # Edge supervision ramps from 0.20 to 1.00 in epochs 1-10; all other lambdas remain fixed.
        edge_warmup_epochs=10,
        edge_warmup_start=0.20,
        resume_checkpoint=resume_checkpoint,
        negative_ratio=None,
        edge_negative_weight=1.0,
        lambda_temporal=0.0,
        lambda_temporal_part=0.0,
        lambda_node=1.0,
        lambda_edge=1.0,
        lambda_dynamic=0.0,
        grad_clip=1.0,
        device='cuda',
        shuffle=True,
        num_parts=7,
        ablation='none',
    )

print(f'Database: {database_output}')
print(f'Run output: {run_output_root}')


Database: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\3rscan_official_static_database_v2
Run output: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_spatial_gated_384_v4


In [2]:
from mtp_pipeline.train import train

# Fresh 100-epoch run. The 384-D object pathway remains unchanged from the baseline.
args = make_train_args()
checkpoint_path = train(args)
print(f'Training finished: {checkpoint_path}')


Label protocol: OCRL-3DSSG official subset: 160 objects, 26 positive multi-label relations
Using official 3DSSG split.
Training on 3852 scenes (Validation reserved: 548)
Computing Alpha class weights to balance rare 3DSSG relationships...


Epoch 1/100: 100%|█| 3852/3852 [06:56<00:00,  9.24it/s, total=6.14, representation=4.2, temporal=0, temporal_part=0, no
Epoch 2/100: 100%|█| 3852/3852 [06:54<00:00,  9.30it/s, total=5.07, representation=3.5, temporal=0, temporal_part=0, no
Epoch 3/100: 100%|█| 3852/3852 [06:50<00:00,  9.39it/s, total=4.64, representation=3.19, temporal=0, temporal_part=0, n
Epoch 4/100: 100%|█| 3852/3852 [06:58<00:00,  9.21it/s, total=4.38, representation=3.01, temporal=0, temporal_part=0, n
Epoch 5/100: 100%|█| 3852/3852 [07:19<00:00,  8.76it/s, total=4.15, representation=2.85, temporal=0, temporal_part=0, n
Epoch 6/100: 100%|█| 3852/3852 [06:56<00:00,  9.24it/s, total=3.98, representation=2.74, temporal=0, temporal_part=0, n
Epoch 7/100: 100%|█| 3852/3852 [06:54<00:00,  9.29it/s, total=3.88, representation=2.68, temporal=0, temporal_part=0, n
Epoch 8/100: 100%|█| 3852/3852 [06:41<00:00,  9.59it/s, total=3.79, representation=2.63, temporal=0, temporal_part=0, n
Epoch 9/100: 100%|█| 3852/3852 [06:39<00

KeyboardInterrupt: 

In [4]:
# Use this cell only after an interruption. Restart the kernel, run the setup cell, then run this cell.
import re
from mtp_pipeline.train import train

checkpoints = list((run_output_root / 'checkpoints').glob('integration_by_parts_3rscan_epoch_*.pt'))
assert checkpoints, 'No checkpoint exists yet. Run the fresh-training cell first.'
resume_checkpoint = max(checkpoints, key=lambda p: int(re.search(r'epoch_(\d+)$', p.stem).group(1)))
print(f'Resuming from: {resume_checkpoint.name}')

checkpoint_path = train(make_train_args(resume_checkpoint=resume_checkpoint))
print(f'Training finished: {checkpoint_path}')


Resuming from: integration_by_parts_3rscan_epoch_75.pt
Label protocol: OCRL-3DSSG official subset: 160 objects, 26 positive multi-label relations
Using official 3DSSG split.
Training on 3852 scenes (Validation reserved: 548)
Computing Alpha class weights to balance rare 3DSSG relationships...
Warm-starting from checkpoint: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_spatial_gated_384_v4\checkpoints\integration_by_parts_3rscan_epoch_75.pt
Resuming at epoch 76 of 100.
Restored optimizer state from checkpoint.


Epoch 76/100: 100%|█| 3852/3852 [07:24<00:00,  8.66it/s, total=1.99, representation=1.96, temporal=0, temporal_part=0, 
Epoch 77/100: 100%|█| 3852/3852 [07:18<00:00,  8.78it/s, total=1.99, representation=1.95, temporal=0, temporal_part=0, 
Epoch 78/100: 100%|█| 3852/3852 [07:21<00:00,  8.72it/s, total=1.98, representation=1.95, temporal=0, temporal_part=0, 
Epoch 79/100: 100%|█| 3852/3852 [07:21<00:00,  8.72it/s, total=1.98, representation=1.95, temporal=0, temporal_part=0, 
Epoch 80/100: 100%|█| 3852/3852 [07:21<00:00,  8.73it/s, total=1.98, representation=1.94, temporal=0, temporal_part=0, 
Epoch 81/100: 100%|█| 3852/3852 [07:47<00:00,  8.23it/s, total=1.97, representation=1.94, temporal=0, temporal_part=0, 
Epoch 82/100: 100%|█| 3852/3852 [07:56<00:00,  8.08it/s, total=1.97, representation=1.94, temporal=0, temporal_part=0, 
Epoch 83/100: 100%|█| 3852/3852 [07:53<00:00,  8.13it/s, total=1.97, representation=1.93, temporal=0, temporal_part=0, 
Epoch 84/100: 100%|█| 3852/3852 [08:23<0

Saved checkpoint: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_spatial_gated_384_v4\checkpoints\integration_by_parts_3rscan_epoch_100.pt
Saved history: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_spatial_gated_384_v4\training_history_3rscan.json
Training finished: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_spatial_gated_384_v4\checkpoints\integration_by_parts_3rscan_epoch_100.pt


In [5]:
import argparse
from mtp_pipeline.evaluate_3dssg import evaluate_model

checkpoints = sorted(
    (run_output_root / 'checkpoints').glob('integration_by_parts_3rscan_epoch_*.pt'),
    key=lambda path: int(path.stem.rsplit('_', 1)[-1]),
)
assert checkpoints, 'No completed checkpoint found.'
checkpoint_path = checkpoints[-1]

results = evaluate_model(argparse.Namespace(
    checkpoint=checkpoint_path,
    reference_root=paths.reference_root,
    output_root=run_output_root,
    database=database_output,
    mode='static',
    train_scans=train_scans_file,
    val_scans=val_scans_file,
    split_seed=SEED,
    device='cuda',
    max_scenes=None,
))
results


Protocol: OCRL-3DSSG official subset: 160 objects, 26 positive multi-label relations
Evaluating 548 official validation scenes on cuda.


Evaluating official OCRL 3DSSG protocol: 100%|███████████████████████████████████████| 548/548 [29:12<00:00,  3.20s/it]


--- Official OCRL/3DSSG Protocol Results (percent) ---
{
    "protocol": "OCRL-3DSSG official subset: 160 objects, 26 positive multi-label relations",
    "object_classes": 160,
    "positive_relation_classes": 26,
    "evaluated_scenes": 548,
    "base_paper_table_2": {
        "Object": {
            "R@1": 55.04,
            "R@5": 75.14,
            "mR@1": 18.83,
            "mR@5": 40.49
        },
        "Predicate": {
            "R@1": 76.26,
            "R@3": 92.09,
            "mR@1": 35.17,
            "mR@3": 57.61
        },
        "Triplet": {
            "R@50": 86.83,
            "R@100": 88.7,
            "mR@50": 49.17,
            "mR@100": 59.4
        }
    },
    "base_paper_table_3": {
        "SGCls": {
            "with_graph_constraints": {
                "R@20": 28.82,
                "R@50": 30.16,
                "R@100": 30.24
            },
            "without_graph_constraints": {
                "R@20": 29.34,
                "R@50": 33.68,
     